In [1]:
words = open("names.txt","r").read().splitlines()

In [2]:
stoi = {ch: i for i,ch in enumerate(sorted(set(".".join(words))))}
itos = {i:ch for ch,i in stoi.items()}

In [3]:
import torch

In [4]:
context = []
context_len = 3
iy = []
for w in words:
    w = w+'.'
    cntxt = [0]*context_len
    for ch in w:
        context.append(cntxt)
        iy.append(stoi[ch])
        cntxt = cntxt[1:] + [stoi[ch]]
context = torch.tensor(context)
iy = torch.tensor(iy)
ix = context
context.shape, iy.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [5]:
emb_len = 10
context_len = 3

In [128]:
g = torch.Generator().manual_seed(2147483647)
C = torch.rand((27, emb_len),dtype=torch.float32,generator = g)

W1 = torch.randn((emb_len*context_len, 300), dtype=torch.float32,generator = g)
b1 = torch.randn((300,), dtype=torch.float32,generator = g)

W2 = torch.randn((300, 27), dtype=torch.float32,generator = g)
b2 = torch.randn((27,), dtype=torch.float32,generator = g)

parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True
sum(p.nelement() for p in parameters)

17697

In [129]:
li = torch.linspace(-3,0,1000)
le = 10**li

lossi = []
ll = []


In [130]:
import torch.nn.functional as F
g = torch.Generator().manual_seed(2147483647)
for i in range(200000):

    batch = torch.randint(0, ix.shape[0],(32,),generator=g)
    emb = C[ix[batch]]

    h = torch.tanh(emb.view(-1, context_len * emb_len) @ W1 + b1)
    logits = h @ W2 + b2

    loss  = F.cross_entropy(logits, iy[batch])
    print(loss.item())

    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if i < 100000 else 0.01

    ll.append(i)
    lossi.append(loss.item())
    for p in parameters:
        p.data += -lr * p.grad

36.902835845947266
23.24827003479004
22.286373138427734
19.224790573120117
19.634214401245117
13.197131156921387
11.112728118896484
11.353395462036133
13.60593032836914
13.970449447631836
12.853100776672363
12.283703804016113
10.43388843536377
14.689522743225098
13.2151517868042
14.948450088500977
14.746405601501465
10.179780960083008
10.54527473449707
9.532793998718262
10.718530654907227
12.304368019104004
8.95018482208252
10.21493148803711
8.51396656036377
8.551481246948242
10.659202575683594
9.31014633178711
10.354366302490234
5.622757434844971
6.901853084564209
8.077301025390625
6.710910797119141
9.82129955291748
9.873096466064453
6.154296875
10.594221115112305
8.497542381286621
9.632611274719238
10.46158504486084
11.620197296142578
8.083556175231934
8.715118408203125
6.786461353302002
5.345335960388184
6.727251052856445
8.45105266571045
6.67626953125
4.866948127746582
6.832273483276367
7.065821647644043
5.891578674316406
7.089947700500488
4.72564172744751
5.744960308074951
5.22574

In [131]:
batch = torch.randint(0, ix.shape[0],(32,),generator=g)
emb = C[ix]

h = torch.tanh(emb.view(-1, context_len * emb_len) @ W1 + b1)
logits = h @ W2 + b2

loss  = F.cross_entropy(logits, iy)
print(loss.item())

2.1794087886810303


In [173]:
with torch.no_grad():
    context_len = 3
    emb_len = 10
    g = torch.Generator().manual_seed(2147483647)
    for i in range(20):
        out = ""
        context = [0]*context_len
        while True:
            emb = C[context]
            h = torch.tanh(emb.view(-1,emb_len*context_len) @ W1 + b1)
            logits = h @ W2 + b2

            prob = torch.softmax(logits, dim = 1)
            ch = torch.multinomial(prob, num_samples = 1,generator=g).item()
            context = context[1:] + [ch]
            if( ch == 0):
                break
            out += itos[ch]


        print(out)

tex
mariah
makila
kayda
vin
mitta
nolla
kama
aristaivaubraxsi
gota
miclie
luvo
kerred
avareyde
sadel
tiaviyah
folstihi
jennatarias
dasor
breenley


In [165]:
context = [0,0,0]

emb = C[context]
h = torch.tanh(emb.view(-1,30) @ W1 + b1)
logits = h @ W2 + b2

prob = torch.softmax(logits, dim = 1)
ch = torch.multinomial(prob, num_samples = 1,generator=g).item()
context = context[1:] + [itos[ch]]
print(context)
out += itos[ch]


[0, 0, 'm']
